# Phase 4: Research-Grounded RAG Layer Demonstration

## Music Brain Wellbeing Intelligence System

This notebook demonstrates the end-to-end **Research Retrieval Layer (RAG Foundation)**:

$$\text{Research JSONL} \rightarrow \text{Document Chunker} \rightarrow \text{Embedding Model} \rightarrow \text{ChromaDB Store} \rightarrow \text{Semantic Retrieval} \rightarrow \text{Evidence Package}$$

> **Scientific Boundary Notice:** This RAG layer provides scientific context from verified PubMed publications regarding music therapy, stress recovery, and emotion regulation. It does **not** make clinical predictions or diagnose mental health conditions.

In [1]:
import os
import sys
import json

# Ensure project root is in python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.rag.chunker import DocumentChunker
from src.rag.embeddings import EmbeddingModel
from src.rag.vector_store import VectorStore
from src.rag.ingest import IngestionPipeline
from src.rag.retriever import ResearchRetriever
from src.rag.adapter import RecommendationQueryAdapter
from src.rag.evidence import build_evidence_package

print("RAG Module imports successful!")

RAG Module imports successful!


## 1. Load Research Corpus Provenance

We load verified PubMed research records from `data/raw/research/music_wellbeing_research.jsonl`.

In [2]:
jsonl_path = os.path.join(project_root, "data", "raw", "research", "music_wellbeing_research.jsonl")

with open(jsonl_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(records)} verified PubMed research papers:")
for r in records:
    print(f" - [{r['year']}] {r['title']} (PMID: {r['pmid']})")

Loaded 10 verified PubMed research papers:
 - [2021] Effects of music therapy on anxiety: A meta-analysis of randomized controlled trials (PMID: 34365216)
 - [2025] Music therapy for the treatment of anxiety: a systematic review with multilevel meta-analyses (PMID: 40547443)
 - [2022] Music therapy for stress reduction: a systematic review and meta-analysis (PMID: 33176590)
 - [2020] Effects of music interventions on stress-related outcomes: a systematic review and two meta-analyses (PMID: 31167611)
 - [2022] Music listening and stress recovery in healthy individuals: A systematic review with meta-analysis of experimental studies (PMID: 35714120)
 - [2024] The effects of music and auditory stimulation on autonomic arousal, cognition and attention: A systematic review (PMID: 38458383)
 - [2024] Scoping Review on the Use of Music for Emotion Regulation (PMID: 39336008)
 - [2007] Personality and music: can traits explain how people use music in everyday life? (PMID: 17456267)
 - [2024] Ne

## 2. Ingestion Pipeline: Chunking, Embedding & ChromaDB Storage

We execute the idempotent ingestion pipeline, encoding documents into 384-dimensional dense vectors using `sentence-transformers/all-MiniLM-L6-v2` and storing them in local ChromaDB vector store.

In [3]:
chroma_dir = os.path.join(project_root, "data", "vector_store", "chroma")
vector_store = VectorStore(persist_directory=chroma_dir, collection_name="music_wellbeing_research")
embedder = EmbeddingModel()
pipeline = IngestionPipeline(embedder=embedder, vector_store=vector_store)

result = pipeline.run(jsonl_path)
print("Ingestion Summary:")
for k, v in result.items():
    print(f"  {k}: {v}")

Ingestion Summary:
  documents_loaded: 10
  chunks_created: 10
  vectors_stored: 10
  status: success
  total_collection_count: 10


## 3. Real Retrieval Demonstrations

We demonstrate 3 real semantic retrieval queries bridging acoustic user profiles and scientific evidence.

In [4]:
retriever = ResearchRetriever(embedder=embedder, vector_store=vector_store)
adapter = RecommendationQueryAdapter()

# Example User Profile: Low energy, slow tempo, high acousticness listener
user_profile = {
    "user_id": "demo_user_01",
    "audio_feature_summary": {
        "energy_mean": 0.32,
        "tempo_mean": 82.5,
        "acousticness_mean": 0.78
    }
}

query_1 = adapter.construct_query_from_profile(user_profile, target_topic="stress reduction autonomic recovery")
print(f"Constructed Research Query 1:\n  '{query_1}'\n")

results_1 = retriever.retrieve(query_1, top_k=2)
for idx, res in enumerate(results_1, 1):
    print(f"--- Result {idx} ---")
    print(f"Title: {res['metadata'].get('title')}")
    print(f"PMID: {res['metadata'].get('pmid')} | Year: {res['metadata'].get('year')}")
    print(f"Similarity Score: {res.get('similarity_score')} | Distance: {res.get('distance'):.4f}")
    print(f"Retrieved Text: {res['text']}\n")

Constructed Research Query 1:
  'stress reduction autonomic recovery low energy soothing acoustics slow tempo rhythmic stability high acousticness instrumental timbre physiological arousal stress recovery individual differences'

--- Result 1 ---
Title: The effects of music and auditory stimulation on autonomic arousal, cognition and attention: A systematic review
PMID: 38458383 | Year: 2024
Similarity Score: 0.142 | Distance: 0.8580
Retrieved Text: A systematic review of 31 empirical studies evaluating the arousal-mood hypothesis, which posits that auditory stimulation (music, white noise, binaural beats) influences cognitive performance via altered autonomic arousal. Reviewing objective physiological markers (cardiac electrophysiology, galvanic skin response, pupillometry), the evidence regarding auditory stimulation effects on autonomic arousal and cognitive task performance was mixed and inconclusive. Current evidence linking acoustic property manipulation directly to cognitive enh

In [5]:
# Query Example 2: Music therapy efficacy for anxiety symptoms
query_2 = "music therapy RCT meta-analysis anxiety symptoms receptive active intervention"
print(f"Constructed Research Query 2:\n  '{query_2}'\n")

results_2 = retriever.retrieve(query_2, top_k=2)
for idx, res in enumerate(results_2, 1):
    print(f"--- Result {idx} ---")
    print(f"Title: {res['metadata'].get('title')}")
    print(f"PMID: {res['metadata'].get('pmid')} | Year: {res['metadata'].get('year')}")
    print(f"Similarity Score: {res.get('similarity_score')} | Distance: {res.get('distance'):.4f}")
    print(f"Retrieved Text: {res['text']}\n")

Constructed Research Query 2:
  'music therapy RCT meta-analysis anxiety symptoms receptive active intervention'

--- Result 1 ---
Title: Effects of music therapy on anxiety: A meta-analysis of randomized controlled trials
PMID: 34365216 | Year: 2021
Similarity Score: 0.2761 | Distance: 0.7239
Retrieved Text: This meta-analysis of 32 randomized controlled trials (RCTs) involving 1,924 participants evaluated the efficacy of music therapy in reducing anxiety. Results demonstrated that music therapy significantly reduced anxiety post-intervention compared to control groups. Subgroup analysis showed positive effects across age groups (<60 and >=60 years) and geographic regions. However, treatment benefits were not maintained at follow-up assessments. Findings highlight short-term efficacy of structured music therapy for anxiety reduction, though clinical durability requires further investigation.

--- Result 2 ---
Title: Music therapy for the treatment of anxiety: a systematic review with 

In [6]:
# Query Example 3: Individual differences and personality traits in music emotion regulation
query_3 = "individual differences personality traits neuroticism openness music emotion regulation"
print(f"Constructed Research Query 3:\n  '{query_3}'\n")

results_3 = retriever.retrieve(query_3, top_k=2)
for idx, res in enumerate(results_3, 1):
    print(f"--- Result {idx} ---")
    print(f"Title: {res['metadata'].get('title')}")
    print(f"PMID: {res['metadata'].get('pmid')} | Year: {res['metadata'].get('year')}")
    print(f"Similarity Score: {res.get('similarity_score')} | Distance: {res.get('distance'):.4f}")
    print(f"Retrieved Text: {res['text']}\n")

Constructed Research Query 3:
  'individual differences personality traits neuroticism openness music emotion regulation'

--- Result 1 ---
Title: Scoping Review on the Use of Music for Emotion Regulation
PMID: 39336008 | Year: 2024
Similarity Score: 0.3863 | Distance: 0.6137
Retrieved Text: A scoping review of 47 studies mapping literature on music emotion regulation (MER). The review highlighted significant conceptual ambiguities between music-induced emotion (passive state) and music emotion regulation (active coping). Most existing literature treats music as a unitary construct without isolating specific intra-musical parameters (acoustic structure, tempo, mode, timbre). Calls for rigorous theoretical modeling of intrinsic acoustic parameters and individual listener intent in emotion regulation strategies.

--- Result 2 ---
Title: Effects of music interventions on stress-related outcomes: a systematic review and two meta-analyses
PMID: 31167611 | Year: 2020
Similarity Score: 0.2611

## 4. Evidence Package Assembly (Phase 4 Endpoint)

We assemble the retrieved chunks and source metadata into a structured `EvidencePackage` payload ready for Phase 5 LLM consumption.

In [7]:
evidence_package = build_evidence_package(
    query=query_1,
    retrieved_chunks=results_1,
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Structured Evidence Package Payload:")
print(json.dumps(evidence_package, indent=2))

Structured Evidence Package Payload:
{
  "query": "stress reduction autonomic recovery low energy soothing acoustics slow tempo rhythmic stability high acousticness instrumental timbre physiological arousal stress recovery individual differences",
  "retrieved_chunks": [
    {
      "chunk_id": "pub_ijpsycho_2024_38458383_chunk_0",
      "text": "A systematic review of 31 empirical studies evaluating the arousal-mood hypothesis, which posits that auditory stimulation (music, white noise, binaural beats) influences cognitive performance via altered autonomic arousal. Reviewing objective physiological markers (cardiac electrophysiology, galvanic skin response, pupillometry), the evidence regarding auditory stimulation effects on autonomic arousal and cognitive task performance was mixed and inconclusive. Current evidence linking acoustic property manipulation directly to cognitive enhancements via physiological arousal remains indirect.",
      "metadata": {
        "chunk_id": "pub_ijps